In [77]:
import pandas as pd 
import numpy as np

imdb = pd.read_csv("../data/raw/IMDB Dataset.csv")
print(imdb["review"].str.contains("<br />").value_counts())
print(imdb[imdb["review"].str.contains("<br />")]["sentiment"].value_counts())
print(imdb.groupby("sentiment")["sentiment"].value_counts())

review
True     29200
False    20800
Name: count, dtype: int64
sentiment
negative    14930
positive    14270
Name: count, dtype: int64
sentiment
negative    25000
positive    25000
Name: count, dtype: int64


In [78]:
# Pattern: < followed by a letter or / and then anything until the next >
tag_pattern = r'<(?:[a-zA-Z/][^>]*|!--.*?--)>'

# Find all matches, flatten the list, and see unique tags
found_tags = imdb["review"].str.findall(tag_pattern).explode().value_counts()

found_tags

review
<br />                                                                       201947
</i>                                                                              8
<i>                                                                               8
<grin>                                                                            2
<p>                                                                               1
<SPOILER>                                                                         1
</SPOILER>                                                                        1
<hr>                                                                              1
<http://rogerebert.suntimes.com/apps/pbcs.dll/section?category=ANSWERMAN>         1
<sp?) classic "Romeo & Juliet". Guess I'll have to rent that next.<br />          1
<sigh>                                                                            1
<em>                                                                 

In [79]:
# Match http... then any non-whitespace... 
# BUT don't include punctuation (.,!?;:) if it's at the very end of the string or before a space.
url_pattern = r'https?://\S+?(?=[.,!?;:\)]?(?:\s|$))'

found_urls = imdb["review"].str.findall(url_pattern).explode().value_counts()

found_urls

review
http://www.PetitionOnline.com/gh1215/petition.html             5
http://blog.myspace.com/locoformovies                          4
http://eattheblinds.blogspot.com/                              3
http://www.invocus.net                                         3
http://ferdinandvongalitzien.blogspot.com/                     3
                                                              ..
http://imdb.com/title/tt0283181/<br                            1
http://friderwaves.com/index.php?page=cave                     1
http://iwascalledclementine.multiply.com/reviews               1
http://www.petitiononline.com/6600F/petition.html              1
http://www.macrophile.com/~arilin/archive/metamorphosis-day    1
Name: count, Length: 110, dtype: int64

In [80]:
from bs4 import BeautifulSoup

def clean_html(text):
    # "lxml" or "html.parser" are common engines
    return BeautifulSoup(text, "html.parser").get_text()

# Apply it to your column
imdb["review_clean"] = imdb["review"].apply(clean_html)

imdb["review_clean"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

In [81]:
# Pattern: < followed by a letter or / and then anything until the next >
tag_pattern = r'<(?:[a-zA-Z/][^>]*|!--.*?--)>'

# Find all matches, flatten the list, and see unique tags
found_tags = imdb["review_clean"].str.findall(tag_pattern).explode().value_counts()

found_tags.sum()


np.int64(0)

In [82]:
# Match http... then any non-whitespace... 
# BUT don't include punctuation (.,!?;:) if it's at the very end of the string or before a space.
url_pattern = r'https?://\S+?(?=[.,!?;:\)]?(?:\s|$))'

found_urls = imdb["review_clean"].str.findall(url_pattern).explode().value_counts()

found_urls

review_clean
http://www.PetitionOnline.com/gh1215/petition.html             5
http://blog.myspace.com/locoformovies                          4
http://www.invocus.net                                         3
http://eattheblinds.blogspot.com/                              3
http://ferdinandvongalitzien.blogspot.com/                     3
                                                              ..
http://imdb.com/title/tt0283181/The                            1
http://friderwaves.com/index.php?page=cave                     1
http://iwascalledclementine.multiply.com/reviews               1
http://www.petitiononline.com/6600F/petition.html              1
http://www.macrophile.com/~arilin/archive/metamorphosis-day    1
Name: count, Length: 109, dtype: int64

In [83]:
imdb.head()

,review,sentiment,review_clean
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. The filming tec...
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is..."


In [84]:
imdb["sentiment"] = imdb["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
imdb.sample(5)
imdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review        50000 non-null  object
 1   sentiment     50000 non-null  int64 
 2   review_clean  50000 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.1+ MB


In [89]:
uppercase_found = imdb["review_clean"].str.findall(r'[A-Z]{2,}').explode().value_counts()
uppercase_found.head(20)

review_clean
TV          5559
DVD         5012
THE         2970
OK          1844
NOT         1293
IMD         1215
US           986
OF           871
II           725
AND          687
CGI          629
IS           629
VHS          570
SPOILERS     483
THIS         480
BBC          464
NO           462
IT           450
UK           414
THAT         394
Name: count, dtype: int64

In [86]:
question_mark_count = imdb["review_clean"].str.findall(r'\?+').explode().value_counts().sum()
excl_mark_count = imdb["review_clean"].str.findall(r'!+').explode().value_counts().sum()
print(question_mark_count, excl_mark_count)

29864 38284


In [87]:
imdb["qm_count"] = imdb["review_clean"].str.findall(r'\?+').str.len()
imdb["em_count"] = imdb["review_clean"].str.findall(r'!+').str.len()
imdb

,review,sentiment,review_clean,qm_count,em_count
0,One of the other reviewers has mentioned that ...,1,One of the other reviewers has mentioned that ...,0,0
1,A wonderful little production. <br /><br />The...,1,A wonderful little production. The filming tec...,0,1
2,I thought this was a wonderful way to spend ti...,1,I thought this was a wonderful way to spend ti...,1,0
3,Basically there's a family where a little boy ...,0,Basically there's a family where a little boy ...,0,2
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,"Petter Mattei's ""Love in the Time of Money"" is...",0,0
...,...,...,...,...,...
49995,I thought this movie did a down right good job...,1,I thought this movie did a down right good job...,0,0
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0,"Bad plot, bad dialogue, bad acting, idiotic di...",0,0
49997,I am a Catholic taught in parochial elementary...,0,I am a Catholic taught in parochial elementary...,0,0
49998,I'm going to have to disagree with the previou...,0,I'm going to have to disagree with the previou...,0,0


In [92]:
all_caps = imdb["review_clean"].str.findall(r'\b[A-Z]{2,}\b')

ignore_set = {
    "IMDB", 
    "IMDb", 
    "SPOILERS AHEAD", 
    "SPOILER AHEAD",
    "SPOILERS",
    "SPOILER",
    "TV", 
    "UK", 
    "BBC", 
    "DVD", 
    "CGI", 
    "VHS", 
    "MGM", 
}

imdb["uppercase_count"] = [
    len([word for word in word_list if word not in ignore_set]) 
    for word_list in all_caps
]
imdb

,review,sentiment,review_clean,qm_count,em_count,uppercase_count
0,One of the other reviewers has mentioned that ...,1,One of the other reviewers has mentioned that ...,0,0,3
1,A wonderful little production. <br /><br />The...,1,A wonderful little production. The filming tec...,0,1,0
2,I thought this was a wonderful way to spend ti...,1,I thought this was a wonderful way to spend ti...,1,0,0
3,Basically there's a family where a little boy ...,0,Basically there's a family where a little boy ...,0,2,2
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,"Petter Mattei's ""Love in the Time of Money"" is...",0,0,0
...,...,...,...,...,...,...
49995,I thought this movie did a down right good job...,1,I thought this movie did a down right good job...,0,0,0
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0,"Bad plot, bad dialogue, bad acting, idiotic di...",0,0,0
49997,I am a Catholic taught in parochial elementary...,0,I am a Catholic taught in parochial elementary...,0,0,0
49998,I'm going to have to disagree with the previou...,0,I'm going to have to disagree with the previou...,0,0,0
